# 00 — EDA (Kaggle, CPU or GPU session)

Goal: look at the data before modelling. We answer four questions:
1. Where inside the 30 frames does the crash happen? (motion curves)
2. Do test clips look different from train (overlays, night, sharpness)? (domain shift)
3. How do twin clips (same `group_id`) overlap?
4. Are there shortcuts (brightness, borders) that separate sources instead of crashes?

**Settings:** attach the competition `cooked-or-not` (Add Input -> Competitions). Accelerator can stay *None* (this notebook is CPU-only). Output: PNGs in `/kaggle/working/eda/`.

In [ ]:
import glob, os
import numpy as np, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

hits = glob.glob("/kaggle/input/**/train_labels.csv", recursive=True) + glob.glob("../../data/train_labels.csv")
assert hits, "train_labels.csv not found - attach the dataset"
ROOT = os.path.dirname(hits[0])
OUT = "/kaggle/working/eda" if os.path.exists("/kaggle/working") else "../../artifacts/eda_local"
os.makedirs(OUT, exist_ok=True)
rng = np.random.default_rng(0)

lab = pd.read_csv(f"{ROOT}/train_labels.csv")
test_ids = pd.read_csv(f"{ROOT}/sample_submission.csv")["clip_id"].tolist()

def load_clip(split, cid):
    d = f"{ROOT}/{split}/{cid}"
    return np.stack([np.asarray(Image.open(f"{d}/frame_{i:03d}.jpg").convert("RGB")) for i in range(30)])

def gray(clip):
    return clip.astype(np.float32).mean(-1)

print(ROOT, len(lab), len(test_ids))

## 1. Label table checks
Expect 1500/1500 and exactly 2 clips per group (1 crash + 1 no-crash).

In [ ]:
print(lab["label"].value_counts().to_string())
g = lab.groupby("group_id")["label"].agg(["count", "sum"])
print(g.value_counts().to_string())
print(f"clips={len(lab)} groups={lab.group_id.nunique()} test={len(test_ids)}")

## 2. Contact sheets
Frames 0, 8, 16, 24, 29 of a few train pairs and test clips. **Look at them**: overlays, night, borders, camera types.

In [ ]:
FR = [0, 8, 16, 24, 29]

def contact_pairs(n_groups=4):
    gids = rng.choice(lab.group_id.unique(), n_groups, replace=False)
    fig, axs = plt.subplots(2 * n_groups, len(FR), figsize=(3 * len(FR), 1.9 * 2 * n_groups))
    r = 0
    for gid in gids:
        for _, row in lab[lab.group_id == gid].sort_values("label").iterrows():
            clip = load_clip("train", row.clip_id)
            for c, f in enumerate(FR):
                axs[r, c].imshow(clip[f]); axs[r, c].axis("off")
                if c == 0: axs[r, c].set_title(f"group {gid} label={row.label}", fontsize=7, loc="left")
            r += 1
    plt.tight_layout(); plt.savefig(f"{OUT}/contact_train_pairs.png", dpi=80); plt.show()

def contact_test(n=8):
    ids = rng.choice(test_ids, n, replace=False)
    fig, axs = plt.subplots(n, len(FR), figsize=(3 * len(FR), 1.9 * n))
    for r, cid in enumerate(ids):
        clip = load_clip("test", cid)
        for c, f in enumerate(FR):
            axs[r, c].imshow(clip[f]); axs[r, c].axis("off")
        axs[r, 0].set_title(cid, fontsize=7, loc="left")
    plt.tight_layout(); plt.savefig(f"{OUT}/contact_test.png", dpi=80); plt.show()

contact_pairs(); contact_test()

## 3. Motion curves and train/test shift statistics
Mean |frame difference| per time step for crash / no-crash / test. A bump in the crash curve tells us **when** the crash happens
(this decides how we crop in time). Brightness / contrast / sharpness histograms show how train and test differ.

In [ ]:
N = 200
pos_ids = lab[lab.label == 1].clip_id.sample(N, random_state=0).tolist()
neg_ids = lab[lab.label == 0].clip_id.sample(N, random_state=0).tolist()
tst_ids = list(rng.choice(test_ids, min(N, len(test_ids)), replace=False))

def stats_for(split, ids):
    diffs, bright, contrast, sharp, mids = [], [], [], [], []
    for cid in ids:
        gr = gray(load_clip(split, cid))
        diffs.append(np.abs(gr[1:] - gr[:-1]).mean((1, 2)))
        m = gr[15]
        bright.append(m.mean()); contrast.append(m.std())
        gy, gx = np.gradient(m); sharp.append(np.sqrt(gx**2 + gy**2).mean())
        mids.append(m)
    return np.array(diffs), np.array(bright), np.array(contrast), np.array(sharp), np.mean(mids, 0)

Dp, Bp, Cp, Sp, Mp = stats_for("train", pos_ids)
Dn, Bn, Cn, Sn, Mn = stats_for("train", neg_ids)
Dt, Bt, Ct, St, Mt = stats_for("test", tst_ids)

In [ ]:
plt.figure(figsize=(9, 4))
for D, name in [(Dp, "train crash"), (Dn, "train no-crash"), (Dt, "test (unlabeled)")]:
    plt.plot(D.mean(0), label=name)
plt.xlabel("t (diff between frame t and t+1)"); plt.ylabel("mean |frame diff|"); plt.legend()
plt.title("Average motion over time"); plt.tight_layout(); plt.savefig(f"{OUT}/diff_curves.png", dpi=100); plt.show()

fig, axs = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (vals, title) in zip(axs, [((Bp, Bn, Bt), "brightness"), ((Cp, Cn, Ct), "contrast"), ((Sp, Sn, St), "sharpness")]):
    allv = np.concatenate(vals); bins = np.linspace(allv.min(), allv.max(), 30)
    ax.hist(np.concatenate(vals[:2]), bins=bins, alpha=0.5, label="train", density=True)
    ax.hist(vals[2], bins=bins, alpha=0.5, label="test", density=True)
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/shift_stats.png", dpi=100); plt.show()

fig, axs = plt.subplots(1, 2, figsize=(10, 3))
axs[0].imshow((Mp + Mn) / 2, cmap="gray"); axs[0].set_title("train: mean middle frame")
axs[1].imshow(Mt, cmap="gray"); axs[1].set_title("test: mean middle frame")
for ax in axs: ax.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/mean_frames.png", dpi=100); plt.show()

print(f"brightness  train={np.concatenate([Bp, Bn]).mean():.1f}  test={Bt.mean():.1f}")
print(f"contrast    train={np.concatenate([Cp, Cn]).mean():.1f}  test={Ct.mean():.1f}")
print(f"sharpness   train={np.concatenate([Sp, Sn]).mean():.2f}  test={St.mean():.2f}")
print(f"mean frame-diff  crash={Dp.mean():.2f}  no-crash={Dn.mean():.2f}  test={Dt.mean():.2f}")

## 4. Twin overlap
Which frames of a crash clip also appear in its no-crash twin? This tells us how the pair was cut from one video
(and why random K-fold would leak).

In [ ]:
def small(clip):
    return np.stack([np.asarray(Image.fromarray(f).resize((96, 56))) for f in gray(clip).astype(np.uint8)]).reshape(30, -1).astype(np.float32)

for gid in rng.choice(lab.group_id.unique(), 5, replace=False):
    rows = lab[lab.group_id == gid]
    if len(rows) != 2 or rows.label.sum() != 1: continue
    A = small(load_clip("train", rows[rows.label == 1].clip_id.iloc[0]))
    B = small(load_clip("train", rows[rows.label == 0].clip_id.iloc[0]))
    match = np.abs(A[:, None] - B[None]).mean(-1) < 2.0
    print(f"group {gid}: crash-clip frames found in twin: {np.where(match.any(1))[0].tolist()} | twin frames matched: {np.where(match.any(0))[0].tolist()}")

Next: write what you notice into `notes/competition.md` -> *EDA findings*, then run `01_extract_features`.